## 1. 이전 과정 회고: End-to-End 연결

지금까지 학습한 내용을 RAG 파이프라인의 각 단계에 매핑하면 다음과 같습니다.

```
                       RAG 파이프라인
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
[1단계: Indexing]           [2단계: Retrieval]         [3단계: Generation]
 문서 → 청킹                    질문 → 임베딩               프롬프트 구성
 → 임베딩                       → 벡터 검색                 → LLM 호출(★이번)
 → 벡터 DB 저장                 → 관련 문서 반환             → 답변 생성
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
```

이번 에는 **Generation 단계** 를 추가하여 완전한 RAG 파이프라인을 구축합니다.

## 2. 수동 RAG 파이프라인 구축 (API 직접 호출)

LangChain을 사용하기 전에, OpenAI API를 직접 호출하여 **순수한 RAG 파이프라인** 을 먼저 구현합니다.

In [ ]:
import os
import numpy as np
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(r'c:\Users\Playdata\Documents\20260528\rag_course\.env')
client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))

# 1. 지식 베이스 문서 (인덱싱 단계)
knowledge_docs = [
    "RAG(Retrieval-Augmented Generation)는 2020년 Meta AI가 발표한 기술로, LLM과 정보 검색을 결합한다. 외부 문서에서 관련 정보를 검색하여 LLM의 컨텍스트로 제공함으로써 답변의 정확성을 높인다.",
    "벡터 데이터베이스는 텍스트 임베딩을 고차원 벡터로 저장하고, 코사인 유사도 기반의 빠른 검색을 지원한다. ChromaDB, FAISS, Pinecone이 대표적이다.",
    "청킹(Chunking)은 긴 문서를 적절한 크기의 조각으로 분할하는 과정이다. 고정 크기 청킹, 문장 기반 청킹, 재귀적 청킹 등의 전략이 있으며, 청크 크기는 검색 품질에 직접적인 영향을 미친다.",
    "임베딩(Embedding)은 텍스트를 의미적 정보를 보존하는 수치 벡터로 변환하는 과정이다. OpenAI의 text-embedding-3-small 모델은 1536차원의 벡터를 생성한다.",
    "LangChain은 LLM 기반 애플리케이션을 구축하기 위한 프레임워크로, 문서 로더, 텍스트 분할기, 임베딩, 벡터 스토어, 체인 등 다양한 컴포넌트를 제공한다.",
    "프롬프트 엔지니어링에서 RAG 시스템의 프롬프트는 시스템 지시사항, 검색된 컨텍스트, 사용자 질문으로 구성된다. 컨텍스트를 명확히 구분하고, 컨텍스트에 없는 내용은 답변하지 않도록 지시하는 것이 중요하다."
]

# 2. 임베딩 생성 (인덱싱 단계)
def get_embedding(text):
    response = client.embeddings.create(input=text, model="text-embedding-3-small")
    return response.data[0].embedding

doc_embeddings = [get_embedding(doc) for doc in knowledge_docs]
print(f"[인덱싱 완료] {len(knowledge_docs)}개 문서의 임베딩 생성 ({len(doc_embeddings[0])}차원)")

# 3. 검색 함수 (Retrieval 단계)
def cosine_similarity(a, b):
    a, b = np.array(a), np.array(b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

def retrieve(query, top_k=3):
    query_emb = get_embedding(query)
    scores = [(i, cosine_similarity(query_emb, emb)) for i, emb in enumerate(doc_embeddings)]
    scores.sort(key=lambda x: x[1], reverse=True)
    return [(knowledge_docs[i], sim) for i, sim in scores[:top_k]]

# 4. 생성 함수 (Generation 단계)
def generate_with_rag(query, retrieved_docs):
    context = "\n\n".join([f"[참고 문서 {i+1}] {doc}" for i, (doc, _) in enumerate(retrieved_docs)])
    
    messages = [
        {"role": "system", "content": "당신은 제공된 참고 문서를 바탕으로 정확하게 답변하는 AI 어시스턴트입니다. 참고 문서에 없는 내용은 '제공된 문서에서 해당 정보를 찾을 수 없습니다'라고 답변하세요."},
        {"role": "user", "content": f"참고 문서:\n{context}\n\n질문: {query}"}
    ]
    
    response = client.chat.completions.create(
        model="gpt-4o-mini", messages=messages, temperature=0
    )
    return response.choices[0].message.content

# 5. 전체 파이프라인 실행
query = "RAG에서 청킹이 왜 중요한가요?"
print(f"\n질문: {query}")
print("=" * 60)

# Retrieval
retrieved = retrieve(query, top_k=3)
print("\n[검색된 문서]")
for i, (doc, sim) in enumerate(retrieved, 1):
    print(f"  {i}위 (유사도: {sim:.4f}): {doc[:60]}...")

# Generation
answer = generate_with_rag(query, retrieved)
print(f"\n[RAG 답변]")
print(answer)

## 3. RAG vs 순수 LLM 비교

RAG를 사용한 경우와 순수 LLM(컨텍스트 없음)을 사용한 경우의 답변 품질을 비교합니다.

In [ ]:
# RAG vs 순수 LLM 비교 실험

def generate_without_rag(query):
    """순수 LLM (컨텍스트 없음)"""
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": query}],
        temperature=0
    )
    return response.choices[0].message.content

# 테스트 질문
test_query = "ChromaDB에서 코사인 유사도 검색을 설정하는 방법을 설명해주세요."

# 순수 LLM
print("=" * 60)
print(f"질문: {test_query}")
print("=" * 60)

print("\n[1] 순수 LLM 응답 (컨텍스트 없음):")
print("-" * 40)
llm_answer = generate_without_rag(test_query)
print(llm_answer[:300] + "..." if len(llm_answer) > 300 else llm_answer)

# RAG
print(f"\n\n[2] RAG 응답 (검색된 문서 기반):")
print("-" * 40)
retrieved = retrieve(test_query, top_k=3)
rag_answer = generate_with_rag(test_query, retrieved)
print(rag_answer[:300] + "..." if len(rag_answer) > 300 else rag_answer)

print(f"\n\n[비교 분석]")
print(f"  순수 LLM 응답 길이: {len(llm_answer)}자")
print(f"  RAG 응답 길이: {len(rag_answer)}자")
print(f"  검색된 참고 문서 수: {len(retrieved)}개")

## 4. LangChain 기반 RAG 체인 구축

이번에는 LangChain의 컴포넌트를 활용하여 더 구조화된 RAG 파이프라인을 구축합니다.

In [ ]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.documents import Document

# 1. 한국어 기술 문서 (더 긴 원본 문서)
raw_documents = [
    Document(page_content="RAG(Retrieval-Augmented Generation)는 2020년 Meta AI 연구팀이 발표한 기술이다. "
        "이 기술은 대규모 언어 모델의 생성 능력과 정보 검색 시스템을 결합한 하이브리드 접근법이다. "
        "기존의 LLM은 학습 데이터에만 의존하여 답변을 생성했기 때문에, 학습 이후의 정보에 대해서는 "
        "정확한 답변을 제공하기 어려웠다. RAG는 질문이 주어지면 먼저 외부 지식 소스에서 관련 문서를 "
        "검색하고, 검색된 문서를 컨텍스트로 활용하여 답변을 생성한다.",
        metadata={"source": "rag_overview", "topic": "RAG"}),
    Document(page_content="벡터 데이터베이스는 RAG 시스템의 핵심 인프라이다. 텍스트 데이터를 고차원 벡터로 변환하여 "
        "저장하고, 코사인 유사도를 기반으로 의미적으로 유사한 문서를 빠르게 검색할 수 있다. "
        "대표적인 벡터 DB로는 ChromaDB, FAISS, Pinecone, Weaviate가 있다. ChromaDB는 설치가 "
        "간편하고 인메모리 모드를 지원하여 프로토타이핑에 적합하다.",
        metadata={"source": "vector_db", "topic": "Vector DB"}),
    Document(page_content="프롬프트 엔지니어링은 LLM의 성능을 최적화하기 위한 핵심 기법이다. RAG 시스템에서의 "
        "프롬프트는 시스템 지시사항, 검색된 컨텍스트, 사용자 질문의 세 부분으로 구성된다. "
        "효과적인 RAG 프롬프트는 컨텍스트를 명확히 구분하고, 컨텍스트에 없는 내용은 답변하지 "
        "않도록 명시적으로 지시해야 한다. 이를 통해 환각을 줄이고 답변의 신뢰성을 높인다.",
        metadata={"source": "prompt_eng", "topic": "Prompt"})
]

# 2. 텍스트 분할 (청킹)
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=150, chunk_overlap=30,
    separators=["\n\n", "\n", ". ", " "]
)
split_docs = text_splitter.split_documents(raw_documents)
print(f"[청킹] 원본 {len(raw_documents)}개 → 분할 {len(split_docs)}개")

# 3. 벡터 스토어 구축
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = Chroma.from_documents(
    documents=split_docs, embedding=embeddings,
    collection_name="langchain_rag_demo"
)
print(f"[벡터 스토어] ChromaDB에 {len(split_docs)}개 청크 저장 완료")

# 4. 검색기 설정
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# 5. RAG 프롬프트 템플릿
rag_prompt = ChatPromptTemplate.from_template(
    """당신은 제공된 참고 문서를 바탕으로 정확하게 답변하는 AI 어시스턴트입니다.
참고 문서에 없는 내용은 '제공된 문서에서 해당 정보를 찾을 수 없습니다'라고 답변하세요.

참고 문서:
{context}

질문: {question}

답변:"""
)

# 6. LLM 설정
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# 7. RAG 체인 실행
query = "RAG 시스템에서 프롬프트를 어떻게 구성해야 하나요?"
print(f"\n질문: {query}")
print("=" * 60)

# 검색
retrieved_docs = retriever.invoke(query)
print(f"\n[검색 결과] {len(retrieved_docs)}개 청크")
for i, doc in enumerate(retrieved_docs, 1):
    print(f"  {i}위: [{doc.metadata.get('topic', 'N/A')}] {doc.page_content[:60]}...")

# 컨텍스트 구성 및 생성
context = "\n\n".join([doc.page_content for doc in retrieved_docs])
formatted_prompt = rag_prompt.format(context=context, question=query)
response = llm.invoke(formatted_prompt)

print(f"\n[RAG 답변]")
print(response.content)

## 5. 다양한 질문으로 RAG 테스트

지식 베이스에 있는/없는 다양한 질문으로 RAG 시스템의 응답을 테스트합니다.

In [ ]:
# 다양한 질문으로 RAG 테스트

test_questions = [
    "ChromaDB의 특징은 무엇인가요?",           # 지식 베이스에 있는 내용
    "RAG는 언제 누가 개발했나요?",              # 지식 베이스에 있는 내용
    "PyTorch와 TensorFlow의 차이는?",           # 지식 베이스에 없는 내용
]

for q in test_questions:
    print(f"\n{'=' * 60}")
    print(f"질문: {q}")
    print("-" * 60)
    
    docs = retriever.invoke(q)
    context = "\n\n".join([d.page_content for d in docs])
    prompt = rag_prompt.format(context=context, question=q)
    resp = llm.invoke(prompt)
    
    print(f"검색된 문서: {len(docs)}개")
    print(f"답변: {resp.content[:200]}")

## 6. RAG 프롬프트 최적화

프롬프트 설계에 따라 RAG 응답의 품질이 크게 달라집니다. 다양한 프롬프트 전략을 비교합니다.

In [ ]:
# 프롬프트 변형 실험

prompt_variants = {
    "기본형": """참고 문서:\n{context}\n\n질문: {question}\n답변:""",
    "역할 부여형": """당신은 AI 기술 전문가입니다. 아래 참고 문서만을 근거로 답변하세요.
참고 문서에 없는 내용은 추측하지 마세요.

참고 문서:
{context}

질문: {question}
답변:""",
    "구조화된 형": """다음 지침에 따라 답변하세요:
1. 반드시 아래 참고 문서의 내용만 사용하세요.
2. 답변은 핵심 내용을 먼저 말하고 세부 사항을 추가하세요.
3. 참고 문서에 없는 내용은 '해당 정보가 문서에 없습니다'라고 하세요.

참고 문서:
{context}

질문: {question}
답변:"""
}

test_q = "RAG 기술의 핵심 원리를 설명해주세요."
docs = retriever.invoke(test_q)
context = "\n\n".join([d.page_content for d in docs])

for name, template in prompt_variants.items():
    print(f"\n{'=' * 60}")
    print(f"[프롬프트: {name}]")
    print("-" * 60)
    
    prompt = ChatPromptTemplate.from_template(template)
    formatted = prompt.format(context=context, question=test_q)
    resp = llm.invoke(formatted)
    
    print(resp.content[:250] + "..." if len(resp.content) > 250 else resp.content)

## 7. 정리 및 핵심 요약

### 핵심 개념 정리

 개별 컴포넌트를 연결하여 **완전한 RAG 파이프라인** 을 구축했습니다.

| 구현 방식 | 특징 | 사용 시나리오 |
|----------|------|-------------|
| 수동 RAG | 각 단계를 직접 구현 | 원리 이해, 커스텀 파이프라인 |
| LangChain RAG | 프레임워크 컴포넌트 활용 | 빠른 프로토타이핑, 모듈식 구성 |

1. **End-to-End RAG**: 인덱싱(청킹→임베딩→저장) → 검색(질의→벡터검색) → 생성(프롬프트→LLM)의 3단계로 구성된다.

2. **RAG vs 순수 LLM**: RAG는 검색된 문서를 근거로 답변하여 환각을 줄이고, 지식 베이스에 없는 질문에 정직하게 답변할 수 있다.

3. **프롬프트 설계**: 역할 부여, 명시적 제한, 구조화된 지침이 RAG 답변의 품질을 크게 좌우한다.

---